In [1]:
import json
import statistics
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
OUTPUT_FOLDER = REPO_ROOT / "results" / "output"
FIGURE_FOLDER = REPO_ROOT / "results" / "figures"
SELECTION = REPO_ROOT / "raw_data" / "supplementary" / "selection.json"
CURRICULUM = "lv_lu"
CURRICULUM_LABEL = "University of Latvia"
VARIANT, NARROW_VARIANT = "overview_learn_takeaways", "overview"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run the notebooks of data_load first"

from cscaf.figures import CourseAdditionFigure, ProgrammePositionFigure
from cscaf.measures import CourseAddition, CurriculumCoverage, Recommender, RoleMapping
from cscaf.storage import Storage
from cscaf.study import Study

study = Study.from_config()
SEED = study.seed
TAU = study.tau
ENDS = study.ends
KEYS = study.curricula
RUN = study.run_id
storage = Storage.of(study)
print("database   :", DB_PATH.relative_to(REPO_ROOT))
print("run        :", RUN)
print("tau        :", TAU, " ends:", ENDS)
print("curriculum :", CURRICULUM, " variant:", VARIANT, " narrow:", NARROW_VARIANT)

database   : database\CSCAF_public.db
run        : sat-12l-sm_all-mpnet-base-v2_414c4542
tau        : 0.585  ends: (0.58, 0.595)
curriculum : lv_lu  variant: overview_learn_takeaways  narrow: overview


In [2]:
mapping = RoleMapping.read(storage)
coverage = CurriculumCoverage(study, storage).statements(KEYS)
statements = sorted(mapping)
assert len(statements) == 150
per_role = {key: CurriculumCoverage.roles(coverage[key]["items"], mapping) for key in KEYS}
names = {role: entry["role_name"] for role, entry in per_role[KEYS[0]].items()}
spread = {role: sorted(per_role[key][role]["coverage"] for key in KEYS) for role in names}
mine = {role: per_role[CURRICULUM][role]["coverage"] for role in names}
covered_by = {key: sum(1 for s in statements if coverage[key]["items"].get(s, {}).get("above_tau")) for key in KEYS}
courses_of_curriculum = storage.curricula.course_count(CURRICULUM)
dedicated = sorted({t["roles"][0] for t in storage.trainings.trainings(RUN)
                    if t["variant"] == VARIANT and len(t["roles"]) == 1})
TARGET = min(dedicated, key=lambda r: mine[r])
order = sorted(names, key=lambda r: mine[r])
print(f"{CURRICULUM}: {courses_of_curriculum} courses, {covered_by[CURRICULUM]} of 150 statements, "
      f"corpus median {statistics.median(covered_by.values())}")
for role in order:
    rank = 1 + sum(1 for v in spread[role] if v > mine[role])
    print(f"  {names[role][:42]:42s} {mine[role]:.3f}  {rank:2d} of 27  "
          f"{'' if role in dedicated else 'no single-role course'}")
print("target role:", TARGET)

lv_lu: 16 courses, 63 of 150 statements, corpus median 75
  Cybersecurity Auditor                      0.250  18 of 27  no single-role course
  Penetration Tester                         0.261  24 of 27  
  Cyber Threat Intelligence Specialist       0.348  18 of 27  
  Digital Forensics Investigator             0.389  24 of 27  
  Cybersecurity Implementer                  0.389  20 of 27  
  Cybersecurity Researcher                   0.417  13 of 27  
  Cyber Legal, Policy & Compliance Officer   0.462  14 of 27  no single-role course
  Cyber Incident Responder                   0.474  19 of 27  
  Cybersecurity Architect                    0.480  17 of 27  
  Cybersecurity Risk Manager                 0.562  14 of 27  no single-role course
  Cybersecurity Educator                     0.706   3 of 27  
  Chief Information Security Officer (CISO)  0.778   9 of 27  
target role: PenTester


In [3]:
target_items = [s for s in statements if TARGET in [r["role"] for r in mapping[s]]]
subsection_of = dict(storage.framework.statement_subsections())
uncovered = [s for s in target_items if not coverage[CURRICULUM]["items"][s]["above_tau"]]
matches = Recommender.best_matches(storage, RUN, CURRICULUM)
recommendation = Recommender(study).recommend(coverage[CURRICULUM]["items"], mapping, TARGET, CURRICULUM, matches)
for s in recommendation.statements:
    print(f"{s.subsection:9s} {s.similarity:.3f} {s.distance_to_tau:.3f}  {s.statement}")

at_ends = {}
for value in ENDS:
    other = study.at_tau(value)
    items = {s: {**entry, "above_tau": entry["similarity"] >= value}
             for s, entry in coverage[CURRICULUM]["items"].items()}
    result = Recommender(other).recommend(items, mapping, TARGET, CURRICULUM, None)
    at_ends[value] = {"covered": sum(1 for s in target_items if items[s]["above_tau"]),
                      "statements": [(s.statement, s.subsection, s.similarity) for s in result.statements]}
    print(f"tau {value:.3f}: covered {at_ends[value]['covered']} of {len(target_items)}")
    for statement in at_ends[value]["statements"]:
        print(f"   {statement[1]:9s} {statement[2]:.4f}  {statement[0]}")

skills    0.518 0.067  Conduct ethical hacking
skills    0.490 0.095  Develop codes, scripts and programmes
knowledge 0.530 0.055  Computer systems vulnerabilities
knowledge 0.522 0.063  Information technology (IT) and operational technology (OT) appliances
tau 0.580: covered 6 of 23
   skills    0.5184  Conduct ethical hacking
   skills    0.4904  Develop codes, scripts and programmes
   knowledge 0.5300  Computer systems vulnerabilities
   knowledge 0.5217  Information technology (IT) and operational technology (OT) appliances
tau 0.595: covered 5 of 23
   skills    0.5184  Conduct ethical hacking
   skills    0.4904  Develop codes, scripts and programmes
   knowledge 0.5927  Cybersecurity attack procedures
   knowledge 0.5300  Computer systems vulnerabilities


In [4]:
role_statements = {}
for role, sid in storage.framework.statement_roles():
    role_statements.setdefault(role, set()).add(sid)
labels_against_content = {"declared first": [], "other role higher": [], "declared none": []}
for course_id, code in storage.trainings.of_variant(VARIANT):
    declared_roles = storage.trainings.declared_roles(course_id)
    covered = storage.similarities.training_statements_above(course_id, TAU)
    shares = {r: len(covered & s) / len(s) for r, s in role_statements.items()}
    declared_best = max(shares[r] for r in declared_roles)
    kind = ("declared none" if declared_best == 0 else
            "declared first" if declared_best == max(shares.values()) else "other role higher")
    labels_against_content[kind].append(code)
print({k: len(v) for k, v in labels_against_content.items()})
for kind, codes in labels_against_content.items():
    print(f"  {kind}: {', '.join(codes)}")

{'declared first': 11, 'other role higher': 8, 'declared none': 3}
  declared first: FOR498, LDR514, LDR519, SEC403, SEC503, SEC504, SEC530, SEC542, SEC560, SEC588, SEC599
  other role higher: FOR578, FOR589, LDR520, LDR521, SEC275, SEC501, SEC565, SEC566
  declared none: FOR518, SEC497, SEC555


In [5]:
base = {rf: entry["similarity"] for rf, entry in coverage[CURRICULUM]["items"].items()
        if entry["similarity"] is not None}
recommended = [s.rf_sentence_id for s in recommendation.statements]
ceilings = CourseAddition.overlap_ceilings(mapping, TARGET)
shared_with_target = {}
for s in target_items:
    for r in mapping[s]:
        if r["role"] != TARGET:
            shared_with_target[r["role"]] = shared_with_target.get(r["role"], 0) + 1
ceiling_on_target = {role: count / len(target_items) for role, count in shared_with_target.items()}

by_variant, sentences_by_variant, titles, declared = {}, {}, {}, {}
for code, variant, rf, score, _ in storage.similarities.training_item_scores(RUN):
    by_variant.setdefault(variant, {}).setdefault(code, {})[rf] = score
for training in storage.trainings.trainings(RUN):
    sentences_by_variant.setdefault(training["variant"], {})[training["code"]] = training["sentences"]
    if training["variant"] == VARIANT:
        titles[training["code"]] = training["title"]
        declared[training["code"]] = training["roles"]


def ceiling_for(code):
    return max((ceiling_on_target.get(role, 0.0) for role in declared[code]), default=0.0)


chosen = json.loads(SELECTION.read_text(encoding="utf-8"))["chosen"]
TARGETS = tuple(sorted(c for c in declared if declared[c] == [TARGET]))
CONTROLS = tuple(sorted((entry["code"] for role, entry in chosen.items()
                         if entry and role != TARGET and declared.get(entry["code"]) == [role]),
                        key=lambda c: (-ceiling_for(c), c)))
ORDER = TARGETS + CONTROLS
changes, checks, unexplained, narrow, ends = {}, {}, {}, {}, {}
addition = CourseAddition(study)
for code in ORDER:
    changes[code] = addition.change(base, by_variant[VARIANT][code], mapping, CURRICULUM, code,
                                    sentences_by_variant[VARIANT][code], recommended)
    checks[code] = CourseAddition.within_ceiling(changes[code], mapping, TARGET)
    unexplained[code] = [role for role, check in checks[code].items()
                         if not check["within"] and role not in declared[code]]
    narrow[code] = addition.change(base, by_variant[NARROW_VARIANT][code], mapping, CURRICULUM, code,
                                   sentences_by_variant[NARROW_VARIANT][code], recommended)
for value in ENDS:
    other = study.at_tau(value)
    ends[value] = {code: CourseAddition(other).change(base, by_variant[VARIANT][code], mapping, CURRICULUM, code,
                                                      sentences_by_variant[VARIANT][code], recommended).delta_for(TARGET)
                   for code in ORDER}
FIGURE_COURSE = max(TARGETS, key=lambda c: (len(changes[c].crossed), changes[c].delta_for(TARGET)))
newly_covered = [mapping[s][0]["statement"] for s in target_items
                 if base.get(s, -1) < TAU <= max(base.get(s, -1), by_variant[VARIANT][FIGURE_COURSE].get(s, -1))]
print(f"targets {TARGETS}\ncontrols {CONTROLS}")
for code in ORDER:
    print(f"{code:7s} {', '.join(declared[code]):18s} {changes[code].sentences_added:3d} "
          f"{changes[code].delta_for(TARGET):+.3f}  ceiling {ceiling_for(code):.3f}  "
          f"crossed {len(changes[code].crossed)}  ends {ends[ENDS[0]][code]:+.3f} {ends[ENDS[1]][code]:+.3f}  "
          f"narrow {narrow[code].delta_for(TARGET):+.3f}  unexplained {unexplained[code] or 'none'}")
print("figure course:", FIGURE_COURSE, "| newly covered:", newly_covered)

targets ('SEC542', 'SEC555', 'SEC560', 'SEC565', 'SEC588')
controls ('SEC503', 'FOR518', 'SEC501', 'FOR498', 'LDR521', 'SEC403', 'SEC497', 'SEC530')
SEC542  PenTester           52 +0.217  ceiling 0.000  crossed 0  ends +0.217 +0.217  narrow +0.174  unexplained none
SEC555  PenTester           31 +0.000  ceiling 0.000  crossed 0  ends +0.000 +0.000  narrow +0.000  unexplained none
SEC560  PenTester           41 +0.174  ceiling 0.000  crossed 0  ends +0.174 +0.130  narrow +0.174  unexplained none
SEC565  PenTester           32 +0.087  ceiling 0.000  crossed 0  ends +0.130 +0.087  narrow +0.087  unexplained none
SEC588  PenTester           28 +0.130  ceiling 0.000  crossed 0  ends +0.130 +0.130  narrow +0.130  unexplained none
SEC503  ThreatIntel         38 +0.043  ceiling 0.304  crossed 0  ends +0.043 +0.043  narrow +0.000  unexplained none
FOR518  IncidentResponder   24 +0.000  ceiling 0.261  crossed 0  ends +0.000 +0.000  narrow +0.000  unexplained none
SEC501  Implementer         36 +

In [6]:
FIGURE_FOLDER.mkdir(parents=True, exist_ok=True)
figures = [ProgrammePositionFigure(spread, mine, names, TARGET, CURRICULUM_LABEL).draw(
               FIGURE_FOLDER / "figure_6_programme_position.png"),
           CourseAdditionFigure(changes[FIGURE_COURSE].before, changes[FIGURE_COURSE].after, ceilings, names,
                                TARGET).draw(FIGURE_FOLDER / "figure_7_course_addition.png")]
for path in figures:
    print("written to", path.relative_to(REPO_ROOT))

written to results\figures\figure_6_programme_position.png
written to results\figures\figure_7_course_addition.png


In [7]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
output = OUTPUT_FOLDER / "06_curriculum_update.json"
record = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "paper_section": "4.2",
    "run": RUN, "tau": TAU, "ends": list(ENDS), "seed": SEED,
    "curriculum": CURRICULUM, "courses": courses_of_curriculum, "statements_covered": covered_by[CURRICULUM],
    "corpus_median": statistics.median(covered_by.values()),
    "role_coverage": mine, "role_order": order, "target_role": TARGET, "roles_with_single_role_course": dedicated,
    "target_uncovered": {"statements": len(uncovered), "of": len(target_items),
                         "skills": sum(1 for s in uncovered if subsection_of[s] == "skills"),
                         "knowledge": sum(1 for s in uncovered if subsection_of[s] == "knowledge")},
    "recommendation": recommendation.as_dict(),
    "recommendation_at_ends": {str(k): v for k, v in at_ends.items()},
    "course_labels_against_content": labels_against_content,
    "target_courses": list(TARGETS), "controls": list(CONTROLS),
    "additions": {code: {"title": titles[code], "declared": declared[code],
                         "sentences": changes[code].sentences_added, "delta": changes[code].delta_for(TARGET),
                         "ceiling": None if code in TARGETS else ceiling_for(code),
                         "crossed": len(changes[code].crossed), "unexplained": unexplained[code],
                         "at_ends": {str(v): ends[v][code] for v in ENDS},
                         "narrow": narrow[code].delta_for(TARGET)} for code in ORDER},
    "figure_course": FIGURE_COURSE, "figure_course_newly_covered": newly_covered,
    "figure_course_other_roles_within_bar": all(check["within"] for role, check in checks[FIGURE_COURSE].items()
                                                if role != TARGET),
    "figures": [str(path.relative_to(REPO_ROOT)) for path in figures],
}
output.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
print("written to", output.relative_to(REPO_ROOT))

written to results\output\06_curriculum_update.json
